# NormLayer + CrewAI Demo

This notebook demonstrates how to use `CrewAIAdapter` to enforce behavioral policies
on task outputs from a CrewAI-style Crew.

**No real CrewAI or LLM calls are needed** — we use mock objects to simulate the crew.

In [ ]:
from normlayer import PolicyEngine
from normlayer.policies.loop_detection import LoopDetection
from normlayer.policies.response_proportionality import ResponseProportionality
from normlayer.adapters.crewai_adapter import CrewAIAdapter

## 1. Define mock CrewAI objects

In a real scenario, these would come from `crewai`. Here we simulate the
`Crew`, `Task`, `Agent`, `TaskOutput`, and `CrewOutput` classes.

In [ ]:
class MockAgent:
    def __init__(self, role):
        self.role = role

class MockTask:
    def __init__(self, agent):
        self.agent = agent

class MockTaskOutput:
    def __init__(self, raw):
        self.raw = raw

class MockCrewOutput:
    def __init__(self, raw, tasks_output):
        self.raw = raw
        self.tasks_output = tasks_output

class MockCrew:
    def __init__(self, tasks, output):
        self.tasks = tasks
        self._output = output
    def kickoff(self, **kwargs):
        return self._output

## 2. Set up the engine and adapter

In [ ]:
engine = PolicyEngine(
    policies=[
        LoopDetection(max_repetitions=2, similarity_threshold=0.9, handler="warn"),
        ResponseProportionality(max_ratio=5.0, handler="warn"),
    ]
)

adapter = CrewAIAdapter(engine)
print("Engine policies:", [p.name for p in engine.policies])

## 3. Clean run — no violations

In [ ]:
researcher = MockAgent(role="researcher")
writer = MockAgent(role="writer")

crew_output = MockCrewOutput(
    raw="Final report complete.",
    tasks_output=[
        MockTaskOutput(raw="Research findings: AI adoption is growing steadily."),
        MockTaskOutput(raw="Blog post draft: AI trends for 2026."),
    ],
)

crew = MockCrew(
    tasks=[MockTask(researcher), MockTask(writer)],
    output=crew_output,
)

wrapped = adapter.wrap(crew)
result = wrapped.kickoff()

print(f"Crew output: {result.raw}")
print(f"Tasks checked: {len(result.tasks_output)}")
print(f"Violations: {len(engine.violations)}")

## 4. Role resolution and agent fallback

When a task has no agent assigned, the adapter falls back to `"crew_agent_{i}"`.

In [ ]:
engine2 = PolicyEngine(
    policies=[LoopDetection(max_repetitions=2, handler="warn")]
)
adapter2 = CrewAIAdapter(engine2)

# Task with no agent assigned
crew_output2 = MockCrewOutput(
    raw="Done",
    tasks_output=[MockTaskOutput(raw="Some output")],
)
crew2 = MockCrew(
    tasks=[MockTask(agent=None)],
    output=crew_output2,
)

wrapped2 = adapter2.wrap(crew2)
wrapped2.kickoff()
print(f"Violations: {len(engine2.violations)}")
print("Agent role fallback works correctly.")

## Summary

The `CrewAIAdapter` wraps `Crew.kickoff()` / `kickoff_async()` and checks each
task output against the policy stack. Agent roles are resolved from the task's
assigned agent, with a clean fallback for unassigned tasks.